In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
url = "https://www.ceneo.pl/91869341#tab=reviews"

In [3]:
response = requests.get(url)
print(f"Status code: {response.status_code}")

Status code: 200


In [4]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [5]:
product_name = page_dom.select_one("h1.product-top__product-info__name").get_text()
print(product_name)

Drukarka HP Color Laser 150nw (4ZB95A)


In [6]:
opinions = page_dom.find_all("div", {'class' : 'js_product-review'})
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
11


In [8]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


In [9]:
extracted_opinions = []

for opinion in opinions:
    
    # Helper function: safely extracts text, returns None if the element is missing
    def get_text(selector):
        element = opinion.select_one(selector)
        return element.text.strip() if element else None

    # Helper function: extracts a list of items (used for pros and cons)
    def get_list(selector):
        elements = opinion.select(selector)
        return [e.text.strip() for e in elements] if elements else []

    # Building the dictionary using your professor's structure
    single_opinion = {
        "opinion_id": opinion.get("data-entry-id"),
        "author": get_text("span.user-post__author-name"),
        "recommendation": get_text("span.user-post__author-recomendation > em"),
        "stars": get_text("span.user-post__score-count"),
        "content": get_text("div.user-post__text"),
        "pros": get_list("div.review-feature__title--positives ~ div.review-feature__item"),
        "cons": get_list("div.review-feature__title--negatives ~ div.review-feature__item"),
        "useful": get_text("button.vote-yes > span"),
        "useless": get_text("button.vote-no > span"),
        "publish_date": get_text("span.user-post__published > time:nth-child(1)"),
        "purchase_date": get_text("span.user-post__published > time:nth-child(2)")
    }
    
    extracted_opinions.append(single_opinion)

# Print the results to verify
print(f"Successfully extracted {len(extracted_opinions)} opinions.")

# Let's look at the first one beautifully formatted
import json
print(json.dumps(extracted_opinions[0], indent=4, ensure_ascii=False))

Successfully extracted 10 opinions.
{
    "opinion_id": "13484305",
    "author": "f...s",
    "recommendation": "Polecam",
    "stars": "4,5/5",
    "content": "Produkt bardzo kompaktowy, co pozwala umieścić go nawet w niewielkich przestrzeniach pod biurkiem (należy pamiętać o wentylacji). Jakość wydruku więcej niż zadowalająca. Drukuje na papierze ozdobnym, ciężkim, na materiałach termotransferowych, etykietach i foliach. Ciężko znaleźć coś na czym nie utrwali wydruku (doskonale radzi sobie z podkładem pod etykiety do transferu na żelowe medium akrylowe (np. nadruki na drewnie bez zdrapywania/rolowania papieru)). Dużą zaletą jest też możliwość taniego nabycia modyfikowanego firmware, które umożliwia drukowanie na zasypce - wtedy strona kolor z papierem, zasypką i prądem oraz amortyzacją samej drukarki wychodzi około 1gr. Ciężko przyczepić się do jakości wydruku. Przy dobrym materiale źródłowym i papierze innym niż \"eko\" jakość jest bardziej niż zadowalająca. Ręczny dupleks również 